# Moving MNIST Dataset Stage

This notebook is the notebook-first data entrypoint for the project. It builds Moving MNIST procedurally from `torchvision.datasets.MNIST`, saves previews under `results/moving_mnist/`, and keeps the tensor contract simple for later JEPA or world-model stages.

**Shape convention**
- Single sample: `(T, C, H, W)`
- DataLoader batch: `(B, T, C, H, W)`

Later world-model or agent stages can treat early frames as context and predict later frames or latent transitions from the same batch tensor.

**Important Colab note**
You do not need to upload any dataset manually. MNIST will be downloaded automatically by `torchvision`.
You do need the repository files to exist inside the Colab runtime. This notebook is configured to clone `https://github.com/Praveen-Rangavajhula/jepa-representation-learning` into `/content` by default if it cannot already find the repo.

In [ ]:
from pathlib import Path
import os
import subprocess

DEFAULT_REPO_URL = 'https://github.com/Praveen-Rangavajhula/jepa-representation-learning.git'

# Option A: point directly at an existing repo in Colab or Drive.
# Example: os.environ['JEPA_REPO_ROOT'] = '/content/jepa-representation-learning'
REPO_ROOT_HINT = os.environ.get('JEPA_REPO_ROOT', '').strip()

# Option B: override the default clone target if needed.
REPO_URL = os.environ.get('JEPA_REPO_URL', DEFAULT_REPO_URL).strip()
REPO_BRANCH = os.environ.get('JEPA_REPO_BRANCH', '').strip()

# Option C: if your repo is on Drive, mount Drive and point JEPA_REPO_ROOT there.
MOUNT_DRIVE = False

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

def looks_like_repo_root(path: Path) -> bool:
    return (path / 'src' / 'jepa').exists() and (path / 'notebooks').exists()

def discover_existing_repo() -> Path | None:
    candidates = []
    if REPO_ROOT_HINT:
        candidates.append(Path(REPO_ROOT_HINT).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    common_roots = [Path('/content'), Path('/content/drive/MyDrive')]
    for root in common_roots:
        if not root.exists():
            continue
        candidates.append(root)
        try:
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    candidates.append(child)
        except Exception:
            pass

    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except Exception:
            continue
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if looks_like_repo_root(candidate):
            return candidate
    return None

def clone_repo_if_needed() -> Path | None:
    if not REPO_URL:
        return None

    repo_name = Path(REPO_URL.rstrip('/')).stem
    if repo_name.endswith('.git'):
        repo_name = repo_name[:-4]
    target_root = Path('/content') / repo_name

    if looks_like_repo_root(target_root):
        print(f'Reusing existing cloned repo: {target_root}')
        return target_root.resolve()

    if target_root.exists() and not looks_like_repo_root(target_root):
        raise FileExistsError(
            f'Target clone directory already exists but does not look like this repo: {target_root}'
        )

    command = ['git', 'clone']
    if REPO_BRANCH:
        command.extend(['--branch', REPO_BRANCH])
    command.extend([REPO_URL, str(target_root)])
    print('Cloning repo:', ' '.join(command))
    subprocess.check_call(command)
    return target_root.resolve()

REPO_ROOT = discover_existing_repo()
if REPO_ROOT is None:
    REPO_ROOT = clone_repo_if_needed()

if REPO_ROOT is None:
    raise FileNotFoundError(
        'The Colab runtime still cannot see this repository. Try one of these options and rerun this cell:\n'
        '1. Leave the defaults alone and make sure outbound git clone works in this Colab session.\n'
        '2. Set os.environ[\'JEPA_REPO_ROOT\'] to an existing repo path in /content or /content/drive/MyDrive.\n'
        '3. If your repo uses a non-default branch, set os.environ[\'JEPA_REPO_BRANCH\'] first.'
    )

SRC_ROOT = REPO_ROOT / 'src'
assert (SRC_ROOT / 'jepa').exists(), f"Expected package directory at {SRC_ROOT / 'jepa'}"

print(f'Repository root: {REPO_ROOT}')
print(f'Source root: {SRC_ROOT}')
%cd $REPO_ROOT

In [ ]:
import importlib.util
import subprocess
import sys

requirements_file = REPO_ROOT / 'requirements-colab.txt'
core_specs = {
    'torch': 'torch>=2.6,<2.8',
    'torchvision': 'torchvision>=0.21,<0.23',
}
missing_core = [name for name in core_specs if importlib.util.find_spec(name) is None]
installed_anything = False

if missing_core:
    command = [sys.executable, '-m', 'pip', 'install', *[core_specs[name] for name in missing_core]]
    print('Installing missing core packages:', missing_core)
    subprocess.check_call(command, cwd=str(REPO_ROOT))
    installed_anything = True
else:
    print('Core packages already available: torch, torchvision')

if requirements_file.exists():
    print(f'Installing notebook extras from {requirements_file.name}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)], cwd=str(REPO_ROOT))
    installed_anything = True
else:
    print('No requirements-colab.txt found; skipping extra installs.')

if installed_anything:
    print('If this cell installed or upgraded core packages, restart the runtime once if later imports behave unexpectedly.')
else:
    print('Environment already satisfied for the dataset stage.')

In [ ]:
import importlib
import sys

if str(SRC_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT.resolve()))

importlib.invalidate_caches()

from jepa.data import (
    MovingMNISTConfig,
    create_moving_mnist_dataloaders,
    save_sample_visualizations,
    summarize_batch,
)
from jepa.data.moving_mnist import describe_dataset_usage

print(f'Using source root: {SRC_ROOT.resolve()}')
print(f'sys.path[0]: {sys.path[0]}')
print(describe_dataset_usage())

In [ ]:
config = MovingMNISTConfig(
    sequence_length=16,
    image_size=64,
    num_digits=2,
    velocity_range=(1.5, 3.5),
    train_size=512,
    test_size=128,
    mnist_root=str(REPO_ROOT / 'data'),
    seed=42,
)

artifacts = create_moving_mnist_dataloaders(
    config,
    batch_size=8,
    num_workers=0,
    pin_memory=False,
)

train_dataset = artifacts['train_dataset']
test_dataset = artifacts['test_dataset']
train_loader = artifacts['train_loader']
test_loader = artifacts['test_loader']

print('Moving MNIST configuration:')
for key, value in config.as_dict().items():
    print(f'- {key}: {value}')
print(f'- train dataset length: {len(train_dataset)}')
print(f'- test dataset length: {len(test_dataset)}')

In [ ]:
from IPython.display import Image as NotebookImage, display

RESULTS_DIR = REPO_ROOT / 'results' / 'moving_mnist'
saved_paths = save_sample_visualizations(
    train_dataset,
    RESULTS_DIR,
    sample_indices=(0, 1, 2),
    max_frames=8,
)

print('Saved preview artifacts:')
for path in saved_paths:
    print(f'- {path}')
    if path.suffix.lower() == '.png':
        display(NotebookImage(filename=str(path)))

In [ ]:
batch = next(iter(train_loader))
summary = summarize_batch(batch)
print(summary)

assert batch.ndim == 5
assert batch.shape[1] == config.sequence_length
assert batch.shape[2] == 1
assert batch.shape[3] == config.image_size
assert batch.shape[4] == config.image_size

print('Tensor shapes verified for later video-model stages.')